# Notebook 4 - the Validation 

In [1]:
#NB4 - Cleanlab validation of the watchlist
#Independent check: does confident-learning flag the same PLR as the hand-built watchlist?
import numpy as np
import pandas as pd
from cleanlab.filter import find_label_issues

/Users/lauragies/Documents/Job/NF_Bootcamp/capstone-kiezkeeper/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#XGBoost OOF probabilities (all 527 PLR) — the watchlist's basis
oof = pd.read_csv("../../models/M_oof_xgboost.csv", dtype={"plr_id": str})
oof["plr_id"] = oof["plr_id"].astype(str).str.zfill(8)

#the hand-built watchlist (25 PLR) — kept separate, used only for the overlap comparison
watchlist = pd.read_csv("../../models/M_watchlist.csv", dtype={"plr_id": str})
watchlist["plr_id"] = watchlist["plr_id"].astype(str).str.zfill(8)

print(f"{len(oof)} PLR total | {int(oof['y_true'].sum())} designated | "
      f"{len(watchlist)} on hand-built watchlist")

527 PLR total | 109 designated | 25 on hand-built watchlist


In [3]:
#Cleanlab needs given labels + out-of-sample pred_probs as an (n, n_classes) matrix
#Class 0 = undesignated, Class 1 = designated; our oof_prob is P(class 1)
labels = oof["y_true"].to_numpy().astype(int)
pred_probs = np.column_stack([1 - oof["oof_prob"], oof["oof_prob"]])  # shape (527, 2)

#find_label_issues: PLR whose given label is inconsistent with the model's confident prediction
#ranked by likelihood of being an issue (most suspicious first)
issue_idx = find_label_issues(
    labels=labels,
    pred_probs=pred_probs,
    return_indices_ranked_by="self_confidence",
)

#assemble the flagged PLR with their info
cl_issues = oof.iloc[issue_idx].copy()
cl_issues["cl_rank"] = np.arange(1, len(cl_issues) + 1)

#we care about issues among UNDESIGNATED PLR (label 0 flagged as looking designated)
#= the confident-learning analog of our early-warning watchlist
cl_undes = cl_issues[cl_issues["y_true"] == 0].copy()

print(f"Cleanlab flagged {len(cl_issues)} label issues total")
print(f"  of which {len(cl_undes)} are undesignated PLR (early-warning analog)")
print(f"  and {len(cl_issues) - len(cl_undes)} are designated PLR flagged as not-designated-looking")

Cleanlab flagged 64 label issues total
  of which 38 are undesignated PLR (early-warning analog)
  and 26 are designated PLR flagged as not-designated-looking


In [4]:
#Overlap: how many of the 25 hand-built watchlist PLR does Cleanlab independently flag?
wl_ids = set(watchlist["plr_id"])
cl_undes_ids = set(cl_undes["plr_id"])

overlap = wl_ids & cl_undes_ids
only_watchlist = wl_ids - cl_undes_ids
only_cleanlab  = cl_undes_ids - wl_ids

print(f"Hand-built watchlist:      {len(wl_ids)} PLR")
print(f"Cleanlab undesignated flags: {len(cl_undes_ids)} PLR")
print(f"Overlap (both agree):        {len(overlap)} PLR "
      f"({len(overlap)/len(wl_ids):.0%} of the watchlist)")
print(f"Only on hand-built watchlist: {len(only_watchlist)}")
print(f"Only flagged by Cleanlab:     {len(only_cleanlab)}")

#which watchlist PLR did Cleanlab NOT flag? (worth inspecting)
if only_watchlist:
    print("\nOn watchlist but not flagged by Cleanlab:")
    print(watchlist[watchlist["plr_id"].isin(only_watchlist)]
          [["plr_name", "bez", "oof_prob", "status3"]].to_string(index=False))

Hand-built watchlist:      25 PLR
Cleanlab undesignated flags: 38 PLR
Overlap (both agree):        25 PLR (100% of the watchlist)
Only on hand-built watchlist: 0
Only flagged by Cleanlab:     13


In [5]:
#which 13 PLR does Cleanlab flag beyond the hand-built 25? (likely ranks 26+)
extra = cl_undes[cl_undes["plr_id"].isin(only_cleanlab)].copy()
extra = extra.merge(
    pd.read_csv("../../data/final_datasets/dataset_0.csv", dtype={"plr_id": str})
      .assign(plr_id=lambda d: d["plr_id"].astype(str).str.zfill(8))[["plr_id", "plr_name", "bez"]],
    on="plr_id", how="left"
)
print(extra[["plr_id", "plr_name", "bez", "oof_prob"]]
      .sort_values("oof_prob", ascending=False).to_string(index=False))

  plr_id            plr_name                             bez  oof_prob
05100316  Carl-Schurz-Straße                    05 - Spandau  0.766517
01100205       Wilhelmstraße                      01 - Mitte  0.756666
04501042    Breitscheidplatz 04 - Charlottenburg-Wilmersdorf  0.712941
12100103     Breitkopfbecken              12 - Reinickendorf  0.712730
09200716 Dörpfeldstraße West           09 - Treptow-Köpenick  0.704122
12100204          Schäfersee              12 - Reinickendorf  0.694774
01100415     Invalidenstraße                      01 - Mitte  0.693884
09100304 Baumschulenweg West           09 - Treptow-Köpenick  0.689616
08100101          Hasenheide                   08 - Neukölln  0.658168
04501151     Hildegardstraße 04 - Charlottenburg-Wilmersdorf  0.638546
04501149  Nikolsburger Platz 04 - Charlottenburg-Wilmersdorf  0.616660
01200624         Heidestraße                      01 - Mitte  0.604541
04501043         Preußenpark 04 - Charlottenburg-Wilmersdorf  0.586272


### Result Discussion 

Cleanlab, applied to the same XGBoost out-of-fold probabilities via confident learning, independently flags all 25 hand-built watchlist PLR as label issues — a 100% overlap. This confirms the watchlist is not an artefact of the chosen rank cutoff: a formal, established method identifies exactly the same core set.

Cleanlab additionally flags 13 further undesignated PLR, all with oof_prob between 0.59 and 0.77 — i.e. just below the N=25 threshold (0.824) — which is consistent with the deliberately conservative cutoff placed at the last visible gap in the probability ranking; below it the probabilities decline smoothly with no natural boundary. 

The two methods therefore agree completely on the core watchlist and differ only at the margin, where the choice of boundary is inherently a judgment call rather than a data-driven one. One caveat: because both approaches rest on the same out-of-fold probabilities, Cleanlab validates the selection rule, not the underlying probabilities — it is a strong consistency check, not a fully independent second source.